# NLP Practical 9 — Information Retrieval

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** inverted index construction, Boolean retrieval, VSM (TF-IDF) and BM25 ranking, LSI via truncated SVD with a note on embedding-based dense retrieval, and Precision/Recall/F1/MAP evaluation.


In [ ]:
!pip install nltk scikit-learn rank_bm25 -q
import nltk
nltk.download("punkt")
nltk.download("stopwords")


In [ ]:
# ============================================================
# PART A: Inverted index + Boolean retrieval
# ============================================================
from collections import defaultdict
import nltk

documents = {
    1: "The quick brown fox jumps over the lazy dog",
    2: "Never jump over a lazy dog quickly",
    3: "A fox is a wild animal that lives in forests",
    4: "The dog barked at the fox in the yard",
}

def tokenize(text):
    return [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]

inverted_index = defaultdict(set)
for doc_id, text in documents.items():
    for term in tokenize(text):
        inverted_index[term].add(doc_id)

def boolean_and(term1, term2):
    return inverted_index[term1] & inverted_index[term2]

print("Postings list for 'fox':", inverted_index["fox"])
print("Postings list for 'lazy':", inverted_index["lazy"])
print("Boolean query 'fox AND lazy':", boolean_and("fox", "lazy"))


In [ ]:
# ============================================================
# PART B: Vector Space Model (TF-IDF) and BM25 ranking
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

corpus = list(documents.values())
doc_ids = list(documents.keys())
query = "lazy fox"

# --- TF-IDF / VSM ---
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
query_vec = vectorizer.transform([query])
vsm_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

print("VSM (TF-IDF cosine similarity) ranking:")
for doc_id, score in sorted(zip(doc_ids, vsm_scores), key=lambda x: -x[1]):
    print(f"  doc{doc_id}: {score:.3f}")

# --- BM25 (the default ranker inside Elasticsearch / Solr / OpenSearch) ---
tokenized_corpus = [tokenize(d) for d in corpus]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(tokenize(query))

print("\nBM25 ranking:")
for doc_id, score in sorted(zip(doc_ids, bm25_scores), key=lambda x: -x[1]):
    print(f"  doc{doc_id}: {score:.3f}")


In [ ]:
# ============================================================
# PART C: LSI (via truncated SVD) and a simple embedding-based semantic search
# ============================================================
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=42)
lsi_doc_vectors = svd.fit_transform(tfidf_matrix)
lsi_query_vector = svd.transform(query_vec)

lsi_scores = cosine_similarity(lsi_query_vector, lsi_doc_vectors).flatten()
print("LSI (latent semantic) ranking:")
for doc_id, score in sorted(zip(doc_ids, lsi_scores), key=lambda x: -x[1]):
    print(f"  doc{doc_id}: {score:.3f}")

print("\nNote: this notebook uses TF-IDF + SVD as a lightweight, offline stand-in for")
print("dense neural embeddings (e.g. sentence-transformers) which need a downloaded")
print("pretrained model. In production RAG systems, this step uses real embedding models,")
print("stored and searched via a vector database (FAISS / Pinecone / pgvector).")


In [ ]:
# ============================================================
# PART D: Precision, Recall, F1, MAP
# ============================================================
relevant_docs = {"fox_query": {1, 2, 3, 4}}       # ground truth relevance judgments
retrieved_docs = [1, 3, 2, 4]                      # system's ranked output, best-first

def precision_recall_f1(retrieved, relevant):
    retrieved_set = set(retrieved)
    tp = len(retrieved_set & relevant)
    precision = tp / len(retrieved_set) if retrieved_set else 0
    recall = tp / len(relevant) if relevant else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return precision, recall, f1

def average_precision(retrieved, relevant):
    hits, sum_prec = 0, 0.0
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant:
            hits += 1
            sum_prec += hits / i
    return sum_prec / len(relevant) if relevant else 0

p, r, f1 = precision_recall_f1(retrieved_docs, relevant_docs["fox_query"])
ap = average_precision(retrieved_docs, relevant_docs["fox_query"])

print(f"Precision: {p:.2f}  Recall: {r:.2f}  F1: {f1:.2f}")
print(f"Average Precision (contributes to MAP across queries): {ap:.2f}")
